# Modules and Libraries

No serious Python program is written from scratch. The language ships with a **standard library** of ready-made modules, and the scientific Python ecosystem adds a second layer built specifically for numerical computation and visualization. Every quantum technology library in existence is built on top of this ecosystem.

This notebook accompanies **Lesson 5** of *Python Programming for Quantum Technology I*. Topics covered:
- How imports work: `import`, `as`, `from ... import`
- The `math` library in depth
- The `random` library for quantum simulation
- First look at NumPy, Matplotlib, SciPy, and SymPy

---
## 1. How Imports Work

In [ ]:
# Basic import: access contents with dot notation
import math

print(math.pi)       # 3.141592653589793
print(math.sqrt(2))  # 1.4142135623730951

In [ ]:
# Import with an alias: universal conventions in scientific Python
import numpy as np
import matplotlib.pyplot as plt

print(np.sqrt(2))   # same result, numpy's version works on arrays too

In [ ]:
# Import specific names: clean for a small number of well-known names
from math import pi, sqrt, exp

print(pi)       # no "math." prefix needed
print(sqrt(2))
print(exp(1))

### Choosing the Right Form

| Form | When to use |
|------|-------------|
| `import module` | General purpose; origin of every name is explicit |
| `import module as alias` | Frequently used packages (`np`, `plt`, `sp`) |
| `from module import name` | A small number of specific names used repeatedly |
| `from module import *` | Avoid; floods the namespace and hides where names come from |

---
## 2. The `math` Library

The `math` module provides mathematical functions for real numbers. Here is the part of the library that matters most for quantum computation.

In [ ]:
import math

print(f"pi  = {math.pi}")
print(f"e   = {math.e}")
print(f"tau = {math.tau}")
print(f"inf = {math.inf}")

In [ ]:
# Powers and logarithms
print(math.sqrt(2))          # 1.4142...
print(math.exp(1))           # 2.7182...   e^1
print(math.log(math.e))      # 1.0         natural log
print(math.log(1000, 10))    # 3.0         log base 10
print(math.log2(8))          # 3.0         log base 2

In [ ]:
# Trigonometry: all functions take radians
print(math.sin(math.pi / 2))   # 1.0
print(math.cos(0))             # 1.0
print(math.tan(math.pi / 4))   # ~1.0

# Convert between radians and degrees
print(math.degrees(math.pi))   # 180.0
print(math.radians(90))        # 1.5707... (pi/2)

In [ ]:
# Use math.isclose() to compare floats for equality
a = math.sqrt(2)**2
print(a)              # 2.0000000000000004  (floating-point error)
print(a == 2.0)       # False  (direct comparison fails)
print(math.isclose(a, 2.0))   # True

### Quantum Application: Phase Angles and Normalization

In [ ]:
import math

def polar_form(z):
    """Return (modulus, phase in degrees) for a complex number z."""
    r = abs(z)
    theta_deg = math.degrees(math.atan2(z.imag, z.real))
    return r, theta_deg

# Four common qubit amplitudes and their polar forms
amplitudes = {
    "|0⟩": 1.0 + 0j,
    "|1⟩": 0.0 + 1j,
    "|+⟩": 1/math.sqrt(2) + 0j,
    "phase90": 1/math.sqrt(2) * (1 + 1j) / math.sqrt(2),
}

print(f"{'State':<10} {'|z|':>8} {'Phase':>10} {'|z|²':>10}")
print("-" * 42)
for label, z in amplitudes.items():
    r, theta = polar_form(z)
    print(f"{label:<10} {r:>8.4f} {theta:>9.1f}° {r**2:>10.4f}")

### Exercise 2.1

The energy levels of a hydrogen atom are given by:

$$E_n = -\frac{13.6 \text{ eV}}{n^2}, \quad n = 1, 2, 3, \ldots$$

Write a function `hydrogen_energy(n)` using `math` functions that:
1. Returns the energy $E_n$ in electron-volts.
2. Also returns the photon wavelength emitted when transitioning from level $n$ to level 1, using $E = hc/\lambda$ with $h = 4.136 \times 10^{-15}$ eV·s and $c = 3 \times 10^8$ m/s.
3. Returns both as a tuple.

Print the energy and wavelength (in nm) for $n = 2, 3, 4$. These are the Balmer series lines.

In [ ]:
import math

# Your code here


---
## 3. The `random` Library

The `random` module generates pseudo-random numbers. Seeding with `random.seed(n)` makes any simulation reproducible.

In [ ]:
import random
random.seed(42)

print(random.random())             # uniform float in [0.0, 1.0)
print(random.uniform(-1.0, 1.0))   # uniform float in [a, b]
print(random.randint(0, 9))        # random int in [a, b] (inclusive)
print(random.choice([0, 1]))       # pick one item from a list

data = ["q0", "q1", "q2", "q3", "q4"]
print(random.sample(data, k=3))    # 3 items without replacement

random.shuffle(data)               # shuffle in place
print(data)

In [ ]:
import random
random.seed(0)

# Gaussian random numbers: model shot noise and decoherence fluctuations
mu = 0.0      # mean
sigma = 0.05  # standard deviation (5% noise)

noisy_probs = [0.7 + random.gauss(mu, sigma) for _ in range(8)]
print("Noisy probability estimates:")
for p in noisy_probs:
    print(f"  {p:.4f}")

### Simulating Projective Measurement

This function ties together everything from previous lessons: normalization, probability computation, and the cumulative sampling pattern.

In [ ]:
import random
import math

def measure(amplitudes, n_shots=1, seed=None):
    """
    Simulate projective measurements of a quantum state.

    Parameters
    ----------
    amplitudes : list of complex
        State amplitudes. Normalized internally.
    n_shots : int
        Number of measurements to perform.
    seed : int or None
        Random seed for reproducibility.

    Returns
    -------
    list of int
        Indices of the measured basis states.
    """
    if seed is not None:
        random.seed(seed)
    norm = math.sqrt(sum(abs(a)**2 for a in amplitudes))
    probs = [abs(a)**2 / norm**2 for a in amplitudes]
    results = []
    for _ in range(n_shots):
        r = random.random()
        cumulative = 0.0
        for i, p in enumerate(probs):
            cumulative += p
            if r < cumulative:
                results.append(i)
                break
    return results

# Biased qubit: P(|0⟩) = 0.09, P(|1⟩) = 0.91
amps = [0.3 + 0j, 0.954 + 0j]
shots = measure(amps, n_shots=20, seed=42)
print("Outcomes: ", shots)
print(f"Frequency of |1⟩: {shots.count(1)/len(shots):.2f}  (true: {abs(amps[1])**2 / sum(abs(a)**2 for a in amps):.2f})")

### Exercise 3.1

Use the `measure` function above to simulate 1000 shots of measuring the state:

$$|\psi\rangle = \frac{\sqrt{3}}{2}|0\rangle + \frac{1}{2}|1\rangle$$

1. Set `seed=0` for reproducibility.
2. Count the number of times each outcome appears.
3. Compute the empirical probabilities and compare them to the theoretical values ($P(|0\rangle) = 3/4$, $P(|1\rangle) = 1/4$).
4. Use `math.isclose` with `abs_tol=0.03` to check whether the empirical probabilities are close to the theoretical ones.

In [ ]:
import math

# Your code here


---
## 4. The Quantum Ecosystem: First Look

The four packages below each get a dedicated lesson later. Run these cells now to see what they can do.

### NumPy: Arrays and Linear Algebra

NumPy provides the `ndarray`: a fast multi-dimensional array built in C. The `@` operator performs matrix multiplication.

In [ ]:
import numpy as np

# State vector for |+⟩ = (1/√2)(|0⟩ + |1⟩)
plus = np.array([1, 1]) / np.sqrt(2)
print("State |+⟩:", plus)

# Hadamard gate as a 2x2 matrix
H = np.array([[1,  1],
              [1, -1]]) / np.sqrt(2)

# Apply gate: matrix-vector multiplication with @
result = H @ plus
print("H|+⟩ =", np.round(result, 10))   # [1. 0.] = |0⟩

# Probability of each outcome
probs = np.abs(result)**2
print("Probabilities:", probs)

### Matplotlib: Visualization

Matplotlib produces plots. Here is the probability distribution for a 3-qubit equal superposition.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

n_qubits = 3
n_states = 2**n_qubits
labels = [f"|{format(k, f'0{n_qubits}b')}⟩" for k in range(n_states)]
probs  = [1 / n_states] * n_states

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(labels, probs, color="steelblue", edgecolor="white")
ax.set_xlabel("Basis state")
ax.set_ylabel("Probability")
ax.set_title("Equal superposition: 3 qubits")
ax.set_ylim(0, 0.25)
ax.axhline(1/n_states, color="tomato", linestyle="--", linewidth=1, label=f"1/{n_states}")
ax.legend()
plt.tight_layout()
plt.show()

### SciPy: Scientific Algorithms

SciPy provides eigenvalue decomposition, optimization, and matrix exponentiation. The example below finds the energy levels of a simple two-level quantum system.

In [ ]:
import numpy as np
from scipy import linalg

# Pauli Z Hamiltonian: eigenvalues are the energy levels
Z = np.array([[1,  0],
              [0, -1]], dtype=float)

eigenvalues, eigenvectors = linalg.eigh(Z)

print("Energy levels (eigenvalues):", eigenvalues)
print("Ground state (lowest energy):")
print(" ", eigenvectors[:, 0])   # first column = lowest eigenvalue
print("Excited state:")
print(" ", eigenvectors[:, 1])

### SymPy: Symbolic Mathematics

SymPy keeps expressions exact. Use it to derive analytical results and verify identities.

In [ ]:
from sympy import symbols, cos, sin, exp, I, pi, simplify, sqrt, Rational

theta, phi = symbols("theta phi", real=True)

# General single-qubit state on the Bloch sphere
alpha = cos(theta / 2)
beta  = exp(I * phi) * sin(theta / 2)

# Verify normalization symbolically (result must be exactly 1)
norm_sq = simplify(alpha.conjugate() * alpha + beta.conjugate() * beta)
print("Norm squared:", norm_sq)   # 1

# Evaluate at theta = pi/2, phi = 0  (the |+⟩ state)
plus_alpha = alpha.subs([(theta, pi/2), (phi, 0)])
plus_beta  = beta.subs([(theta, pi/2), (phi, 0)])
print(f"alpha at theta=pi/2: {plus_alpha} = {simplify(plus_alpha)}")
print(f"beta  at theta=pi/2: {plus_beta}  = {simplify(plus_beta)}")

### Exercise 4.1

Use NumPy to verify that the Hadamard gate is **unitary**: a matrix $U$ is unitary if $U^\dagger U = I$, where $U^\dagger$ is the conjugate transpose.

1. Define the Hadamard matrix `H` as a NumPy array.
2. Compute `H.conj().T` (the conjugate transpose).
3. Compute the matrix product `H.conj().T @ H`.
4. Use `np.allclose` to check whether the result is the identity matrix `np.eye(2)`.

Then repeat the check for the Pauli X gate `[[0, 1], [1, 0]]`.

In [ ]:
import numpy as np

# Your code here


---
## Summary

| Library | Import convention | Primary use in quantum technology |
|---------|------------------|-----------------------------------|
| `math` | `import math` | Constants, trig, log, `isclose` |
| `random` | `import random` | Simulating measurements, shot noise |
| NumPy | `import numpy as np` | State vectors, gates, matrix multiplication |
| Matplotlib | `import matplotlib.pyplot as plt` | Probability distributions, histograms |
| SciPy | `from scipy import linalg` | Eigenvalues, optimization, matrix exponential |
| SymPy | `from sympy import ...` | Symbolic derivations, exact identities |

**Key rules for imports:**
- Prefer `import module` or `import module as alias`.
- Use `from module import name` for a small number of well-known names.
- Never use `from module import *` in code you care about.
- Always use `math.isclose` (or `np.allclose`) instead of `==` when comparing floats.

**What comes next:** The next lesson introduces **object-oriented programming**. Understanding classes and objects is essential for working with quantum frameworks like Qiskit and PennyLane, which represent circuits, registers, and backends as objects.

---
## Challenge Problem

**Comparing a simulation to theory across shot counts.**

For the state $|\psi\rangle = \frac{1}{\sqrt{3}}|0\rangle + \sqrt{\frac{2}{3}}|1\rangle$:

1. Compute the exact theoretical probabilities using `math`.
2. Using the `measure` function defined earlier in this notebook, simulate measurements for each shot count in `[10, 50, 100, 500, 1000, 5000]` with `seed=0`.
3. For each shot count, compute the empirical probability of $|1\rangle$ and the absolute error versus the theoretical value.
4. Store the results in a list of tuples `(n_shots, empirical_prob, error)`.
5. Plot the absolute error vs. shot count using Matplotlib. Label the axes. Add a horizontal dashed line at `error = 0.01`.

What pattern do you observe as shot count increases?

In [ ]:
import math
import matplotlib.pyplot as plt

# State amplitudes
amps = [1/math.sqrt(3), math.sqrt(2/3)]

# Your code here
